In [ ]:
import subprocess, re, json

def list_devices():
    out = subprocess.run(["arp", "-a"], capture_output=True, text=True).stdout
    devices = []
    for ip, mac in re.findall(r"(\d+\.\d+\.\d+\.\d+)\s+([0-9a-fA-F]{2}(?:-[0-9a-fA-F]{2}){5})", out):
        if mac.lower() == "ff-ff-ff-ff-ff-ff" or ip.startswith(("224.", "239.")) or ip.endswith(".255"):
            continue
        devices.append({"ip": ip, "mac": mac})
    return json.dumps(devices)

print(list_devices())

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def ping(ip):
    subprocess.run(["ping", "-n", "1", "-w", "300", ip], capture_output=True)

def list_devices(scan=True):
    if scan:
        with ThreadPoolExecutor(64) as ex:
            ex.map(ping, [f"192.168.1.{i}" for i in range(1, 255)])
    out = subprocess.run(["arp", "-a"], capture_output=True, text=True).stdout
    devices = []
    for ip, mac in re.findall(r"(\d+\.\d+\.\d+\.\d+)\s+([0-9a-fA-F]{2}(?:-[0-9a-fA-F]{2}){5})", out):
        first = int(ip.split(".")[0])
        if mac.lower() == "ff-ff-ff-ff-ff-ff" or 224 <= first <= 239 or ip.endswith(".255"):
            continue
        devices.append({"ip": ip, "mac": mac})
    return json.dumps(devices)

print(list_devices())

In [ ]:
import psutil
from collections import Counter

def my_connections(limit=20):
    conns = []
    for c in psutil.net_connections(kind="inet"):
        if c.status == "ESTABLISHED" and c.raddr:
            try:
                name = psutil.Process(c.pid).name() if c.pid else "?"
            except Exception:
                name = "?"
            conns.append({"process": name, "remote_ip": c.raddr.ip, "remote_port": c.raddr.port})
    return json.dumps({
        "total_connections": len(conns),
        "by_program": Counter(c["process"] for c in conns).most_common(10),
        "sample": conns[:limit]
    })

print(my_connections())


In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=" key")
MODEL_NAME = "claude-sonnet-5"

def my_connections(limit=20):
    conns = []
    for c in psutil.net_connections(kind="inet"):
        if c.status == "ESTABLISHED" and c.raddr and not (c.raddr.ip.startswith("127.") or c.raddr.ip == "::1"):
            try: name = psutil.Process(c.pid).name() if c.pid else "?"
            except Exception: name = "?"
            conns.append({"process": name, "remote_ip": c.raddr.ip, "remote_port": c.raddr.port})
    return json.dumps({"total": len(conns),
                       "by_program": Counter(c["process"] for c in conns).most_common(10),
                       "sample": conns[:limit]})

TOOL_FUNCS = {"list_devices": list_devices, "my_connections": my_connections}
tools = [
    {"name": "list_devices", "description": "Scans the local home network (192.168.1.x) and returns devices with IP and MAC addresses.",
     "input_schema": {"type": "object", "properties": {}}},
    {"name": "my_connections", "description": "Returns this computer's active internet connections: which programs connect to which remote IPs and ports.",
     "input_schema": {"type": "object", "properties": {"limit": {"type": "integer"}}}},
]

SYSTEM = """You are a home network security assistant. Use the tools, then answer in short, clear Saudi Arabic.
Notes: 192.168.1.1 is usually the router. A MAC whose second hex character is 2, 6, A or E is randomized,
usually a phone using private Wi-Fi addressing. Port 443 is normal HTTPS. Flag anything unusual calmly,
without exaggerating, and give practical advice."""

def run_agent(question):
    messages = [{"role": "user", "content": question}]
    while True:
        resp = client.messages.create(model=MODEL_NAME, max_tokens=2000, system=SYSTEM, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                print(f"🔧 يستخدم: {b.name}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": TOOL_FUNCS[b.name](**b.input)})
        messages.append({"role": "user", "content": results})

print("✅ إيجنت شبكتك جاهز")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(run_agent("مين متصل على شبكتي؟ وهل فيه شي غريب في اتصالات جهازي؟")))

In [ ]:
%pip install scapy

In [ ]:
import json
from collections import Counter, defaultdict
from scapy.all import PcapReader, IP, TCP, UDP, ICMP, ARP, DNSQR

def analyze_pcap(path=r"C:\Users\narje\my_traffic.pcap", max_packets=200000):
    total, first, last = 0, None, None
    proto, src = Counter(), Counter()
    syn_ports, syn_count = defaultdict(set), Counter()
    icmp, dns, arp_map = Counter(), Counter(), defaultdict(set)

    with PcapReader(path) as reader:
        for p in reader:
            total += 1
            t = float(p.time); first = first or t; last = t
            if ARP in p and p[ARP].op == 2:
                proto["ARP"] += 1; arp_map[p[ARP].psrc].add(p[ARP].hwsrc)
            if IP in p:
                s, d = p[IP].src, p[IP].dst; src[s] += 1
                if TCP in p:
                    proto["TCP"] += 1
                    flags = int(p[TCP].flags)
                    if flags & 0x02 and not flags & 0x10:
                        syn_count[s] += 1; syn_ports[(s, d)].add(p[TCP].dport)
                elif UDP in p: proto["UDP"] += 1
                elif ICMP in p: proto["ICMP"] += 1; icmp[s] += 1
                if p.haslayer(DNSQR):
                    dns[p[DNSQR].qname.decode(errors="ignore").rstrip(".")] += 1
            if total >= max_packets: break

    duration = max((last or 0) - (first or 0), 1)
    return json.dumps({
        "packets": total, "seconds": round(duration), "packets_per_sec": round(total / duration, 1),
        "protocols": dict(proto), "top_talkers": src.most_common(5),
        "top_dns_domains": dns.most_common(10),
        "possible_port_scans": [{"src": s, "dst": d, "ports_tried": len(ps)}
                                for (s, d), ps in syn_ports.items() if len(ps) >= 30],
        "top_syn_senders": syn_count.most_common(3),
        "top_icmp_senders": icmp.most_common(3),
        "arp_spoofing_suspects": {ip: list(m) for ip, m in arp_map.items() if len(m) > 1},
    })

print(analyze_pcap())

In [ ]:
TOOL_FUNCS["analyze_pcap"] = analyze_pcap
tools.append({
    "name": "analyze_pcap",
    "description": "Analyzes a Wireshark capture file of the home network. Returns packet rate, protocols, top talkers, "
                   "top DNS domains, and attack indicators: port scans, SYN floods, ICMP floods, ARP spoofing.",
    "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}}
})
print("✅ صار عند الإيجنت", len(tools), "أدوات")

In [ ]:
display(Markdown(run_agent("حلل الترافيك اللي سجلته في Wireshark، فيه أي علامات هجوم؟ وليش فيه طلبات لـ baidu؟")))

In [ ]:
tools[-1]["description"] += " The user's capture file is C:\\Users\\narje\\my_traffic.pcap — omit 'path' to use it."

def run_agent(question):
    messages = [{"role": "user", "content": question}]
    while True:
        resp = client.messages.create(model=MODEL_NAME, max_tokens=2000, system=SYSTEM, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                print(f"🔧 يستخدم: {b.name} {b.input}")
                try:
                    output = TOOL_FUNCS[b.name](**b.input)
                except Exception as e:
                    output = json.dumps({"error": str(e)})
                    print(f"⚠️ خطأ رجع للإيجنت: {e}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": output})
        messages.append({"role": "user", "content": results})

print("✅ الإيجنت صار أذكى")

In [ ]:
display(Markdown(run_agent("حلل الترافيك اللي سجلته في Wireshark، فيه أي علامات هجوم؟ وليش فيه طلبات لـ baidu؟")))